# Premier tests

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100 80GB PCIe


In [106]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("HuggingFaceTB/SmolLM3-3B")
print(config.hidden_size)
print(config.num_attention_heads)
print(config.num_hidden_layers)

2048
16
36


## Avec vllm pour l'inférence

- dtype:"bfloat16" : 
    - Brain Float 16 : moitier mem que float32 , même range dynamique
    - 2 byte / param -> 3B*2 ~ 6 gb
- gpu_memory_utilization=0.1 : limite l'utilisation du gpu à 10% ~ 8gb
- max_model_len=4096 : limite la taille des buffers du KV cache

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="HuggingFaceTB/SmolLM3-3B",
    dtype="bfloat16",
    gpu_memory_utilization=0.1,
    max_model_len=4096
)

In [ ]:
prompt = "What is the capital of France ? The first word of your response should be the answer"
params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=200)

outputs = llm.generate(prompt, params)

print(outputs[0].outputs[0].text)
print(outputs[0].outputs[0].finish_reason)

## Avec AutoModelCausalLM pour l'inférence

In [1]:
from residualstream_utils import load_model, tokenize_prompt

model_name = "HuggingFaceTB/SmolLM3-3B"
device = "cuda"

model, tokenizer = load_model(model_name, device)

2026-03-18 16:15:01.591 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [56]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128012,
  "pad_token_id": 128004,
  "temperature": 0.6,
  "top_p": 0.95
}



In [57]:
model.generation_config.num_beams

1

SmolLM3 peut utiliser un mode de réflexion long (enable_thinking = True)

In [2]:
inputs = tokenize_prompt("The capital of paris is", tokenizer, model, is_chat=True, enable_thinking=True)

generated_ids = model.generate(**inputs, max_new_tokens=100)
output_ids = generated_ids[0][len(inputs.input_ids[0]) :]
print(tokenizer.decode(output_ids, skip_special_tokens=True))


<think>
Okay, the user asked, "The capital of Paris is..." Hmm, that's a bit confusing. Paris is a city in France, and the capital of France is actually Paris. But the way the question is phrased seems a bit off. Maybe they're trying to ask what the capital of Paris is, but that doesn't make sense because Paris is the capital of France, not another country.

Wait, maybe there's a misunderstanding here. Let me think. If someone says


## Test : récupérer les hidden states du modèle, print partir de quelle couche la réponse apparait

In [ ]:
from residualstream_utils import print_top_tokens_per_layer

print_top_tokens_per_layer("The capital of France is", "Paris", model, tokenizer)

Paris est détecté parmis les top 20 token seulement à la couche 31/36

In [ ]:
prompt_tokenized = tokenize_prompt(
                        "The capital of France is", 
                        tokenizer, 
                        model, 
                        is_chat=False, 
                        prompt_system="Continue the sentence", 
                        add_gen_prompt=True, 
                        is_thinking=False
                    )

print_top_tokens_per_layer(prompt_tokenized, "Paris", model, tokenizer, is_already_tokenized=True)

## Généraliser le test avec différent prompts (les pays et capitales du monde) + vizu

In [76]:
df.to_csv("country_capital_prompts.csv")

In [4]:
from residualstream_utils import generate_country_capital_prompts

df_path = "data/countries.csv"

df = generate_country_capital_prompts(df_path)
df.head()

,Prompt,Label
0,The capital of Afghanistan is,Kabul
1,The capital of Albania is,Tirana
2,The capital of Algeria is,Algiers
3,The capital of Andorra is,Andorra la Vella
4,The capital of Angola is,Luanda


In [54]:
from residualstream_vizualisation import plot_layer_analysis

y  = plot_layer_analysis(model, tokenizer, df, top_k=10, min_proba_threshold=0.01, 
                    comp_text=False, comp_token=True, verif_sec_token=False)

100%|██████████| 196/196 [00:16<00:00, 11.66it/s]


## Test tokenization de Copenhagen

In [10]:
text = "Copenhagen"
tokens = tokenizer.encode(text)
print(f"{text} token IDs: {tokens}")

for t in tokens:
    print(f"{t} -> \"{tokenizer.decode([t])}\"")

Copenhagen token IDs: [34, 45929]
34 -> "C"
45929 -> "openhagen"


## Vizu des TOP 5 tokens par couche

In [10]:
import numpy as np
import plotly.express as px
import pandas as pd
from residualstream_utils import remove_spaces, tokenize_prompt, get_hidden_states
from tqdm import tqdm
import torch

def verify_second_token(model, inputs, first_token_id, second_token_id):
    new_input_ids = torch.cat(
        [inputs["input_ids"], torch.tensor([[first_token_id]], device=inputs["input_ids"].device)],
        dim=1
    )

    with torch.no_grad():
        outputs = model(input_ids=new_input_ids)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    pred_token_id = torch.argmax(probs, dim=-1).item()

    return pred_token_id == second_token_id

def top_k_per_layer(model, tokenizer, hidden_states, label, layers, top_k, min_proba_threshold, comp_text, comp_token, verif_sec_token):
    return_top_k = np.zeros((layers[1]-layers[0]+1, top_k))
    for i, h in enumerate(hidden_states[layers[0]:layers[1]+1], start=0):

        last_token_hidden = h[:, -1, :]
        logits = model.lm_head(last_token_hidden)
        probs = torch.softmax(logits, dim=-1)

        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)

        for j, tid in enumerate(top_token_ids[0], start=0):
            decoded_token = tokenizer.decode([tid.item()])
            label_ids = tokenizer.encode(label)[0]
            encoded_label = label_ids[0]
            # if(tid.item() == encoded_label or remove_spaces(decoded_token) == remove_spaces(label)):
            #     return_top_k[i, j] += 1
            #     #print(f"Label {label} trouvé pour pays {i}, TOP {j} !")
            

            if(comp_text):
                if(remove_spaces(decoded_token) == remove_spaces(label)):
                    prob = top_probs[0][j].item()
                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    return_top_k[i, j] += 1
                
            # Comparaison token ids
            if(comp_token):
                if(tid.item() == encoded_label):
                    prob = top_probs[0][j].item()

                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    
                    if(len(label_ids) == 1 or not verif_sec_token):
                        return_top_k[i, j] += 1

                    # Multi token labels
                    elif(verif_sec_token and verify_second_token(model, inputs, label_ids[0], label_ids[1])):
                        return_top_k[i, j] += 1

    return return_top_k
    

def topk_analysis(model, tokenizer, df, layers, top_k, min_proba_threshold, comp_text=True, comp_token=True, verf_sec_token=True):

    nb_layers = layers[1] - layers[0] +1

    global_topk_cout = np.zeros((nb_layers, top_k))

    for row in tqdm(df.itertuples(), total=len(df)):
        #print(f"Prompt : {row.Prompt}, Label : {row.Label}")
        inputs = tokenize_prompt(row.Prompt, tokenizer, model, is_chat=True, prompt_system="Answer with one word.")
        hidden_states = get_hidden_states(model, inputs)
        topk_count = top_k_per_layer(model, tokenizer, hidden_states, row.Label, layers=layers, top_k=top_k,
                                    min_proba_threshold=min_proba_threshold, comp_text=comp_text, comp_token=comp_token, verif_sec_token=verf_sec_token)

        global_topk_cout += topk_count

    return global_topk_cout, layers, top_k


def plot_topk_analysis(data, layers, top_k):

    x = np.arange(layers[0], layers[1]+1)

    # convert to long format
    df = pd.DataFrame(data, columns=[f"TOP {i+1}" for i in range(top_k)])
    df["Layers"] = x
    df = df.melt(id_vars="Layers", var_name="Token", value_name="Number of correct token")

    fig = px.bar(
        df,
        x="Layers",
        y="Number of correct token",
        color="Token",
        barmode="group",
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig.show()

In [11]:
data, layers, top_k = topk_analysis(model, tokenizer, df, layers=(25,36), top_k=5, min_proba_threshold=0.01 ,comp_text=True)

  0%|          | 0/196 [00:00<?, ?it/s]


TypeError: 'int' object is not subscriptable

In [63]:
plot_topk_analysis(data, layers, top_k)

## Heatmap


In [ ]:
import plotly.express as px

def plot_heatmap(model, tokenizer, prompt, label, top_k=200):
    hs = get_hidden_states_from_raw_prompt(model, tokenizer, prompt)
    logit_len = np.zeros(len(hs))

    for i, h in tqdm(enumerate(hs, start=0), total=len(hs)):
        last_token_hidden = h[:, -1, :]
        logits = model.lm_head(last_token_hidden)
        probs = torch.softmax(logits, dim=-1)

        #print(f"Taille probs : {len(probs[0])}")
        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)


        for j, tid in enumerate(top_token_ids[0], start=0):
            decoded_token = tokenizer.decode([tid.item()])
            encoded_label = tokenizer.encode(label)[0]
 
            if(tid.item() == encoded_label or remove_spaces(decoded_token) == remove_spaces(label)):
                #print(f"LAYER {i} tid ({tid.item()}) : \"{decoded_token}\"")
                if top_probs[0][j] > logit_len[i]:
                    logit_len[i] = top_probs[0][j]
    

    heatmap_data = logit_len.reshape(1, -1)

    fig = px.imshow(
        heatmap_data,
        x=np.arange(len(logit_len)),
        y=[label],
        color_continuous_scale="Viridis",
        labels=dict(x="Layer", y="Token", color="Probability"),
        title=f" Probability of '{label}' across layers for prompt : '{prompt}'",
        zmin=0,
        zmax=1
    )
    fig.update_layout(
        height=400
    )

    fig.show()

    return logit_len
    

In [99]:
logit_len = plot_heatmap(model, tokenizer, "The capital of Spain is", "Madrid", top_k=500)

  0%|          | 0/37 [00:00<?, ?it/s]

100%|██████████| 37/37 [00:01<00:00, 21.35it/s]


In [ ]:
from residualstream_utils import *

In [2]:
model, tokenizer = load_model()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
print_top_tokens_per_layer(prompt_tokenized, "Paris", model, tokenizer, is_already_tokenized=True)

Layer 30: [" capital": 0.0014, " acknow": 0.0013, " filmy": 0.0012, " uphol": 0.0012, " agre": 0.0012, " breathable": 0.0010, " detain": 0.0010, "Cap": 0.0010, " swirl": 0.0010, " shimmer": 0.0009, "4": 0.0009, " cap": 0.0009, "2": 0.0009, " conspic": 0.0009, "5": 0.0009, " Cap": 0.0009, "Capital": 0.0009, " cuffs": 0.0008, " Capital": 0.0008, "7": 0.0007]
Layer 31: [" agre": 0.0129, " filmy": 0.0098, " acknow": 0.0092, "2": 0.0057, "4": 0.0038, "5": 0.0036, " advertis": 0.0036, "3": 0.0034, "7": 0.0032, " breathable": 0.0032, " detain": 0.0024, " feas": 0.0023, " hurd": 0.0021, "1": 0.0020, " oy": 0.0020, " slugg": 0.0020, " unavoid": 0.0019, "6": 0.0019, " unmist": 0.0019, "8": 0.0019]
Layer 32: ["2": 0.1089, "5": 0.0483, "4": 0.0400, "3": 0.0376, "1": 0.0250, "7": 0.0215, "6": 0.0172, "8": 0.0118, "0": 0.0104, "9": 0.0095, "10": 0.0065, "h": 0.0056, "100": 0.0054, " filmy": 0.0046, " oy": 0.0033, "25": 0.0030, " breathable": 0.0028, "22": 0.0028, "17": 0.0025, "20": 0.0023]
Layer 33

In [4]:
print_top_tokens_per_layer("The capital of France is", "Paris", model, tokenizer)


 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 30: [" Paris": 0.5117, " a": 0.0537, " not": 0.0271, " French": 0.0225, " ": 0.0198, " called": 0.0113, " one": 0.0099, " known": 0.0088, " now": 0.0088, " an": 0.0082, " actually": 0.0078, " D": 0.0068, " M": 0.0064, " [": 0.0059, " also": 0.0057, " the": 0.0053, " L": 0.0052, " “": 0.0050, " located": 0.0047, " named": 0.0043]

 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 31: [" Paris": 0.9570, " a": 0.0100, " not": 0.0027, " ": 0.0027, " one": 0.0024, " French": 0.0022, " known": 0.0022, " an": 0.0008, " called": 0.0008, " in": 0.0007, ",": 0.0006, " also": 0.0006, " France": 0.0006, " located": 0.0006, " (": 0.0005, " M": 0.0005, " [": 0.0004, " the": 0.0004, " now": 0.0004, " D": 0.0003]

 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 32: [" Paris": 0.5391, " a": 0.1553, " ": 0.1064, " known": 0.0393, " (": 0.0186, " not": 0.0186, ",": 0.0144, " an": 0.0099, " in": 0.0099, " one": 0.0077, " [": 0.0064, " "": 0.0047, ":": 0.0041,

## Test recup couches cachées avec generate

In [1]:
from residualstream_utils import load_model

In [2]:
model, tokenizer = load_model()

2026-03-19 14:56:37.306 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
country = "Denmark"
prompt = f"What is the capital city of {country}?"
messages = [
    {"role": "system",
        "content": (
            f"You are an expert geographer. "
            f"You have to give name of the capital city of this country : {country}. "
            f"Answer only with the capital name "
            f"without any other words or repetition of the question. Don't repeat the prompt neither. "
            f"Example of answer: 'Paris'."
        )
    },
    {"role": "user", "content": prompt}
]
# Tokenize input prompt

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([inputs], return_tensors="pt").to("cuda")


In [ ]:
# outputs = model.generate(
#     **inputs,
    
#     do_sample=True,
#     temperature=0.3,
#     max_new_tokens=10,
#     # top_p=0.9,
#     # top_k=50,
#     output_hidden_states =True,
#     return_dict_in_generate=True,
#     output_scores=True,
#     # repetition_penalty=1.2
# )
model_outputs = model.generate(**model_inputs, return_dict_in_generate=True, output_hidden_states=True)

In [ ]:
generated_ids = model_outputs.sequences[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

Copenhagen


In [ ]:
len(generated_ids.hidden_states[0]) # -> 1er step de generation (premier token ?)

37

In [106]:
step1_lastlayer = generated_ids.hidden_states[0][36]

In [107]:
last_token_hidden = step1_lastlayer[:, -1, :]
logits = model.lm_head(last_token_hidden)
probs = torch.softmax(logits, dim=-1)

top_probs, top_token_ids = torch.topk(probs, 10, dim=-1)
top1 = top_token_ids[0][0]
tokenizer.decode([top1.item()])

'C'

## Plot stackbar


In [ ]:
from residualstream_vizualisation import compute_stackbar_region

df_path = "data/countries.csv"

res_df = compute_stackbar_region(model, tokenizer, df_path)

In [ ]:
from residualstream_vizualisation import plot_stackbar_region

plot_stackbar_region(res_df)

In [ ]:
res_df[res_df["Layer"] == 28]